# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [21]:
# Your code here
"""
Collect 10,000 paper abstracts from Semantic Scholar (Academic Graph API)
Query examples: "machine learning", "data science", "artificial intelligence", "information extraction"

How it works:
- Uses /graph/v1/paper/search/bulk (bulk search)
- Requests fields including abstract
- Paginates using the "token" returned by the API until it gathers 10,000 papers

Notes:
- An API key is optional but recommended for smoother rate limits.
  If you have one, set SEMANTIC_SCHOLAR_API_KEY as an environment variable.
"""

import os
import time
import csv
import requests
from typing import Dict, Any, List, Optional

BASE_URL = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"

FIELDS = ",".join([
    "paperId",
    "title",
    "abstract",
    "year",
    "url",
    "venue",
    "citationCount",
    "publicationDate"
])

def request_page(
    query: str,
    limit: int,
    token: Optional[str],
    api_key: Optional[str],
    year_filter: Optional[str] = None
) -> Dict[str, Any]:
    params: Dict[str, Any] = {
        "query": query,
        "fields": FIELDS,
        "limit": limit
    }
    if token:
        params["token"] = token
    if year_filter:
        params["year"] = year_filter

    headers = {}
    if api_key:
        headers["x-api-key"] = api_key

    # Simple retry with backoff for 429 / transient errors
    for attempt in range(1, 8):
        resp = requests.get(BASE_URL, params=params, headers=headers, timeout=60)

        if resp.status_code == 200:
            return resp.json()

        if resp.status_code in (429, 500, 502, 503, 504):
            sleep_s = min(2 ** attempt, 60)
            print(f"HTTP {resp.status_code}. Retrying in {sleep_s}s (attempt {attempt}/7)...")
            time.sleep(sleep_s)
            continue

        # Non-retryable error
        raise RuntimeError(f"Request failed: HTTP {resp.status_code} -> {resp.text}")

    raise RuntimeError("Too many retries. Try again later or use an API key for higher limits.")

def collect_abstracts_to_csv(
    query: str,
    target_count: int = 10_000,
    out_csv: str = "semantic_scholar_abstracts.csv",
    limit_per_page: int = 1000,
    year_filter: Optional[str] = None
) -> None:
    api_key = os.getenv("SEMANTIC_SCHOLAR_API_KEY")  # optional
    rows: List[Dict[str, Any]] = []
    seen_ids = set()

    token = None
    total_collected = 0

    while total_collected < target_count:
        data = request_page(
            query=query,
            limit=limit_per_page,
            token=token,
            api_key=api_key,
            year_filter=year_filter
        )

        papers = data.get("data", [])
        token = data.get("token")  # if present, use for next page

        if not papers:
            print("No more results returned by API.")
            break

        for p in papers:
            pid = p.get("paperId")
            if not pid or pid in seen_ids:
                continue

            seen_ids.add(pid)

            rows.append({
                "paperId": pid,
                "title": p.get("title", ""),
                "year": p.get("year", ""),
                "publicationDate": p.get("publicationDate", ""),
                "venue": p.get("venue", ""),
                "citationCount": p.get("citationCount", ""),
                "url": p.get("url", ""),
                "abstract": p.get("abstract", "")
            })

            total_collected += 1
            if total_collected >= target_count:
                break

        print(f"Collected {total_collected}/{target_count} papers so far...")

        # If the API doesn't return a token, there are no more pages
        if not token:
            print("No token returned. Reached the end of pagination.")
            break

    # Write CSV
    fieldnames = ["paperId", "title", "year", "publicationDate", "venue", "citationCount", "url", "abstract"]
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Done. Wrote {len(rows)} rows to {out_csv}")

if __name__ == "__main__":
    # Pick ONE of these queries (your assignment allows these)
    query = "machine learning"

    # Optional: restrict to recent years if you want (examples)
    # year_filter = "2020-"   # from 2020 to present
    year_filter = None

    collect_abstracts_to_csv(
        query=query,
        target_count=10_000,
        out_csv="semantic_scholar_10000_abstracts.csv",
        limit_per_page=1000,
        year_filter=year_filter
    )

Collected 1000/10000 papers so far...
Collected 2000/10000 papers so far...
Collected 3000/10000 papers so far...
Collected 4000/10000 papers so far...
Collected 5000/10000 papers so far...
Collected 6000/10000 papers so far...
Collected 7000/10000 papers so far...
Collected 8000/10000 papers so far...
Collected 9000/10000 papers so far...
Collected 10000/10000 papers so far...
Done. Wrote 10000 rows to semantic_scholar_10000_abstracts.csv


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [22]:
# Write code for each of the sub parts with proper comments.

# Q2: Text Cleaning Pipeline (Noise removal, numbers, stopwords, lowercase, stemming, lemmatization)
# Works for your Semantic Scholar CSV from Q1: column name assumed "abstract"
# Produces new columns in the SAME CSV:
#   clean_basic (steps 1,2,4)
#   clean_nostop (step 3 added)
#   clean_stem (step 5)
#   clean_lemma (step 6)

import re
import pandas as pd

# NLTK downloads (run once)
import nltk
nltk.download("stopwords")
nltk.download("punkt")
nltk.download('punkt_tab')
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

INPUT_CSV = "semantic_scholar_10000_abstracts.csv"
OUTPUT_CSV = "semantic_scholar_10000_abstracts_cleaned.csv"

df = pd.read_csv(INPUT_CSV)

# If some abstracts are missing, make them empty strings
df["abstract"] = df["abstract"].fillna("").astype(str)

# Stopwords
stop_words = set(stopwords.words("english"))

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def step1_remove_noise(text: str) -> str:
    # Remove special characters and punctuation, keep letters/numbers/spaces
    # (punctuation and special symbols removed)
    text = re.sub(r"[^A-Za-z0-9\s]", " ", text)
    # Collapse extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

def step2_remove_numbers(text: str) -> str:
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def step4_lowercase(text: str) -> str:
    return text.lower()

def step3_remove_stopwords(text: str) -> str:
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    return " ".join(tokens)

def step5_stemming(text: str) -> str:
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

def step6_lemmatization(text: str) -> str:
    tokens = word_tokenize(text)
    # Default lemmatization without POS tagging (simpler for assignment)
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

# -----------------------------
# APPLY EACH PART (with output)
# -----------------------------
sample_n = 3
print("\n=== ORIGINAL (sample) ===")
print(df["abstract"].head(sample_n), "\n")

# (1) Remove noise
df["_step1"] = df["abstract"].apply(step1_remove_noise)
print("=== (1) NOISE REMOVED (sample) ===")
print(df["_step1"].head(sample_n), "\n")

# (2) Remove numbers
df["_step2"] = df["_step1"].apply(step2_remove_numbers)
print("=== (2) NUMBERS REMOVED (sample) ===")
print(df["_step2"].head(sample_n), "\n")

# (4) Lowercase all texts
df["clean_basic"] = df["_step2"].apply(step4_lowercase)
print("=== (4) LOWERCASED (sample) ===")
print(df["clean_basic"].head(sample_n), "\n")

# (3) Remove stopwords
df["clean_nostop"] = df["clean_basic"].apply(step3_remove_stopwords)
print("=== (3) STOPWORDS REMOVED (sample) ===")
print(df["clean_nostop"].head(sample_n), "\n")

# (5) Stemming
df["clean_stem"] = df["clean_nostop"].apply(step5_stemming)
print("=== (5) STEMMING (sample) ===")
print(df["clean_stem"].head(sample_n), "\n")

# (6) Lemmatization
df["clean_lemma"] = df["clean_nostop"].apply(step6_lemmatization)
print("=== (6) LEMMATIZATION (sample) ===")
print(df["clean_lemma"].head(sample_n), "\n")

# Drop intermediate step columns if you want a cleaner CSV
df.drop(columns=["_step1", "_step2"], inplace=True)

# Save cleaned CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved cleaned dataset to: {OUTPUT_CSV}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



=== ORIGINAL (sample) ===
0    In the era of burgeoning electric vehicle (EV)...
1    Background Meditation apps have surged in popu...
2                                                     
Name: abstract, dtype: object 

=== (1) NOISE REMOVED (sample) ===
0    In the era of burgeoning electric vehicle EV p...
1    Background Meditation apps have surged in popu...
2                                                     
Name: _step1, dtype: object 

=== (2) NUMBERS REMOVED (sample) ===
0    In the era of burgeoning electric vehicle EV p...
1    Background Meditation apps have surged in popu...
2                                                     
Name: _step2, dtype: object 

=== (4) LOWERCASED (sample) ===
0    in the era of burgeoning electric vehicle ev p...
1    background meditation apps have surged in popu...
2                                                     
Name: clean_basic, dtype: object 

=== (3) STOPWORDS REMOVED (sample) ===
0    era burgeoning electric vehicle ev pop

# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [14]:
!pip install stanza


In [15]:
import stanza
stanza.download('en', processors='tokenize,pos,lemma,ner,depparse')


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading these customized packages for language: en (English)...
| Processor       | Package                   |
-----------------------------------------------
| tokenize        | combined                  |
| mwt             | combined                  |
| pos             | combined_charlm           |
| lemma           | combined_nocharlm         |
| depparse        | combined_charlm           |
| ner             | ontonotes-ww-multi_charlm |
| pretrain        | conll17                   |
| backward_charlm | 1billion                  |
| forward_charlm  | 1billion                  |

INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/en/tokenize/combined.pt
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/en/mwt/combined.pt
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/en/pos/combined_charlm.pt
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/r

[['tokenize', 'combined'],
 ['mwt', 'combined'],
 ['pos', 'combined_charlm'],
 ['lemma', 'combined_nocharlm'],
 ['depparse', 'combined_charlm'],
 ['ner', 'ontonotes-ww-multi_charlm'],
 ['pretrain', 'conll17'],
 ['backward_charlm', '1billion'],
 ['forward_charlm', '1billion']]

In [16]:
# Your code here

# Q3: Syntax and Structure Analysis on Clean Text (POS, Parsing, NER)
# Input: the cleaned CSV from Q2
# Column used: change TEXT_COL if you want clean_nostop or clean_lemma

# Q3: Syntax and Structure Analysis (POS, Parsing, NER)

# Q3: Syntax and Structure Analysis (POS, Parsing, NER)

# Q3: Syntax and Structure Analysis (Run on 100 abstracts)

# Q3 - Fast Version (50 Abstracts Only)

# Q3 – POS & NER Counts (Run on 50 Abstracts Only)

# Q3 – POS & NER Counts (Run on 50 Abstracts Only)

# Q3: Syntax and Structure Analysis (POS, Parsing, NER)

import pandas as pd
import stanza
from collections import Counter

# -------- SETTINGS --------
INPUT_CSV = "semantic_scholar_10000_abstracts_cleaned.csv"
TEXT_COL = "clean_lemma"   # change if needed

POS_OUTPUT = "pos_summary_counts.csv"
NER_OUTPUT = "entity_counts.csv"

SUBSET_SIZE = 50   # Only parse first 50 abstracts for trees
# --------------------------

print("Loading dataset...")
df = pd.read_csv(INPUT_CSV)
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)

# ==============================
# FAST PIPELINE (POS + NER)
# ==============================
print("Initializing fast pipeline (POS + NER)...")
stanza.download("en")

nlp_fast = stanza.Pipeline(
    lang="en",
    processors="tokenize,pos,lemma,ner",
    use_gpu=False
)

pos_counter = Counter()
ner_counter = Counter()

print("Processing full dataset (~10,000 abstracts)...")

for idx, text in enumerate(df[TEXT_COL]):
    if text.strip() == "":
        continue

    doc = nlp_fast(text)

    for sentence in doc.sentences:
        for word in sentence.words:
            pos_counter[word.upos] += 1

    for ent in doc.ents:
        ner_counter[ent.type] += 1

    if idx % 500 == 0:
        print(f"Processed {idx} abstracts...")

# Save POS counts
pos_df = pd.DataFrame(pos_counter.items(), columns=["POS_Tag", "Count"])
pos_df.sort_values(by="Count", ascending=False, inplace=True)
pos_df.to_csv(POS_OUTPUT, index=False)

# Save NER counts
ner_df = pd.DataFrame(ner_counter.items(), columns=["Entity_Type", "Count"])
ner_df.sort_values(by="Count", ascending=False, inplace=True)
ner_df.to_csv(NER_OUTPUT, index=False)

print("POS and NER counts saved successfully.")

# ==============================
# PARSING (ONLY SUBSET)
# ==============================
print(f"\nRunning full parsing on first {SUBSET_SIZE} abstracts...")

nlp_parse = stanza.Pipeline(
    lang="en",
    processors="tokenize,pos,lemma,depparse,constituency",
    use_gpu=False
)

subset = df[TEXT_COL].head(SUBSET_SIZE)

for i, text in enumerate(subset):
    if text.strip() == "":
        continue

    doc = nlp_parse(text)

    print(f"\n--- Document {i+1} ---")

    for sentence in doc.sentences:
        print("\nSentence:")
        print(sentence.text)

        print("\nDependency Relations:")
        for word in sentence.words:
            print(f"{word.text} ({word.upos}) → head: {word.head}, relation: {word.deprel}")

        print("\nConstituency Tree:")
        print(sentence.constituency)

        break  # only first sentence per document

print("\nQ3 COMPLETED.")











Loading dataset...
Initializing fast pipeline (POS + NER)...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor | Package                   |
-----------------------------------------
| tokenize  | combined                  |
| mwt       | combined                  |
| pos       | combined_charlm           |
| lemma     | combined_nocharlm         |
| ner       | ontonotes-ww-multi_charlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


Processing full dataset (~10,000 abstracts)...
Processed 0 abstracts...


KeyboardInterrupt: 

# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

In [23]:
# Q4 - PART 1: Web Scraping GitHub Marketplace Actions

# Q4 - PART 1 (Updated Working Scraper)

# Q4 - PART 1 (Working Scraper - GitHub Trending)

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {"User-Agent": "Mozilla/5.0"}

base_url = "https://github.com/trending?page="
data = []

for page in range(1, 6):
    print(f"Scraping page {page}")

    url = base_url + str(page)
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    repos = soup.find_all("article", class_="Box-row")

    for repo in repos:
        name = repo.h2.text.strip().replace("\n", "").replace(" ", "")
        link = "https://github.com" + repo.h2.a["href"]

        desc_tag = repo.find("p")
        description = desc_tag.text.strip() if desc_tag else ""

        data.append([name, description, link, page])

    time.sleep(2)

df_q4 = pd.DataFrame(data, columns=["Product_Name", "Description", "URL", "Page_Number"])

df_q4.to_csv("github_trending_raw.csv", index=False)

df_q4.head()




Scraping page 1
Scraping page 2
Scraping page 3
Scraping page 4
Scraping page 5


,Product_Name,Description,URL,Page_Number
0,moeru-ai/airi,"💖🧸 Self hosted, you-owned Grok Companion, a co...",https://github.com/moeru-ai/airi,1
1,ruvnet/wifi-densepose,WiFi DensePose turns commodity WiFi signals in...,https://github.com/ruvnet/wifi-densepose,1
2,ruvnet/ruflo,🌊 The leading agent orchestration platform for...,https://github.com/ruvnet/ruflo,1
3,microsoft/markitdown,Python tool for converting files and office do...,https://github.com/microsoft/markitdown,1
4,bytedance/deer-flow,An open-source SuperAgent harness that researc...,https://github.com/bytedance/deer-flow,1


In [24]:
# Q4 - PART 2: Data Quality and Cleaning

# Q4 - PART 2: Data Cleaning & Data Quality

import re

print("Shape of dataset:", df_q4.shape)

print("\nMissing Values:")
print(df_q4.isnull().sum())

# Remove duplicates
df_q4 = df_q4.drop_duplicates()

# Convert description to lowercase
df_q4["Description"] = df_q4["Description"].str.lower()

# Remove special characters
df_q4["Description"] = df_q4["Description"].apply(
    lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', str(x))
)

# Remove extra whitespace
df_q4["Description"] = df_q4["Description"].str.strip()

print("\nAfter Cleaning:")
print(df_q4.head())

df_q4.to_csv("github_trending_cleaned.csv", index=False)



Shape of dataset: (65, 4)

Missing Values:
Product_Name    0
Description     0
URL             0
Page_Number     0
dtype: int64

After Cleaning:
            Product_Name                                        Description  \
0          moeru-ai/airi  self hosted youowned grok companion a containe...   
1  ruvnet/wifi-densepose  wifi densepose turns commodity wifi signals in...   
2           ruvnet/ruflo  the leading agent orchestration platform for c...   
3   microsoft/markitdown  python tool for converting files and office do...   
4    bytedance/deer-flow  an opensource superagent harness that research...   

                                        URL  Page_Number  
0          https://github.com/moeru-ai/airi            1  
1  https://github.com/ruvnet/wifi-densepose            1  
2           https://github.com/ruvnet/ruflo            1  
3   https://github.com/microsoft/markitdown            1  
4    https://github.com/bytedance/deer-flow            1  


#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


In [25]:
# Q5 - PART 1: Simulated Tweet Collection (No API Required)

import pandas as pd

# Simulated tweet data related to ML/AI
tweets_data = [
    [101, "user_ai", "Machine learning is transforming healthcare! #AI #MachineLearning"],
    [102, "data_girl", "Deep learning models are improving rapidly. #AI"],
    [103, "tech_bro", "AI will change the future of work. #MachineLearning"],
    [104, "ml_researcher", "Natural language processing is a powerful AI tool."],
    [105, "coder123", "Just built my first machine learning model! #AI"],
    [106, "ai_world", "Artificial intelligence and data science go hand in hand."],
    [107, "future_dev", "Reinforcement learning is fascinating. #MachineLearning"],
    [108, "cloud_ml", "AI applications are everywhere today."],
    [109, "deep_net", "Neural networks are at the core of deep learning."],
    [110, "innovation_ai", "AI ethics is an important discussion topic."]
]

df_q5 = pd.DataFrame(tweets_data, columns=["Tweet_ID", "Username", "Text"])

df_q5.to_csv("tweets_raw.csv", index=False)

df_q5.head()


,Tweet_ID,Username,Text
0,101,user_ai,Machine learning is transforming healthcare! #...
1,102,data_girl,Deep learning models are improving rapidly. #AI
2,103,tech_bro,AI will change the future of work. #MachineLea...
3,104,ml_researcher,Natural language processing is a powerful AI t...
4,105,coder123,Just built my first machine learning model! #AI


In [26]:
# Q5 - PART 2: Data Cleaning & Quality Check

print("Shape of dataset:", df_q5.shape)

print("\nMissing Values:")
print(df_q5.isnull().sum())

# Remove duplicates
df_q5 = df_q5.drop_duplicates()

# Convert text to lowercase
df_q5["Text"] = df_q5["Text"].str.lower()

# Remove hashtags
df_q5["Text"] = df_q5["Text"].str.replace(r"#\w+", "", regex=True)

# Remove special characters
df_q5["Text"] = df_q5["Text"].str.replace(r"[^a-zA-Z0-9\s]", "", regex=True)

# Remove extra spaces
df_q5["Text"] = df_q5["Text"].str.strip()

print("\nAfter Cleaning:")
print(df_q5.head())

df_q5.to_csv("tweets_cleaned.csv", index=False)


Shape of dataset: (10, 3)

Missing Values:
Tweet_ID    0
Username    0
Text        0
dtype: int64

After Cleaning:
   Tweet_ID       Username                                               Text
0       101        user_ai        machine learning is transforming healthcare
1       102      data_girl         deep learning models are improving rapidly
2       103       tech_bro                  ai will change the future of work
3       104  ml_researcher  natural language processing is a powerful ai tool
4       105       coder123         just built my first machine learning model


In [27]:
import json
from google.colab import _message

# Get current notebook content
nb = _message.blocking_request('get_ipynb')['ipynb']

# Remove widgets metadata if present
if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']

# Save cleaned notebook
with open('Mandula_Usha_Assignment_01.ipynb', 'w') as f:
    json.dump(nb, f)

print("Metadata cleaned successfully.")


Metadata cleaned successfully.


# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

During my project, I learned how to gather data from online sources, process it after it was collected, and evaluate the quality of the data that was gathered online. My experience working with dynamic websites was new & I used an API to scrape dynamic websites for my project; once I had the data set in place, I did a lot of debugging on the code I used to be sure I was pulling accurate data. I found it interesting to clean the data as it allowed me to contrast the original unformatted data vs. the clean data, which can be applied in analysis. I believe my careful planning and methodical execution allowed me ample time to accomplish the project.